Links scraping

In [ ]:
from datetime import datetime, timedelta
import json
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

BASE = "https://www.bbc.com"
WAYBACK = "https://web.archive.org/web"

def normalize_url(url: str) -> str:
    return url.split("#")[0].split("?")[0]

def collect_article_links(date: str, category: str) -> list[str]:
    # Use Wayback Machine ONLY to find what links existed on that date
    entry_url = f"{WAYBACK}/{date}000000/{BASE}/{category}"

    headers = {"User-Agent": "SchoolAssignmentBot/1.0"}
    try:
        r = requests.get(entry_url, headers=headers, timeout=30)
        r.raise_for_status()
    except Exception as e:
        print(f"Could not fetch archive for {date}: {e}")
        return []

    soup = BeautifulSoup(r.text, "lxml")

    links: set[str] = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin(entry_url, href)

        # Extract the original BBC link from the archive wrapper
        if "bbc.com" in full and "/articles/" in full:
            if "https://www.bbc.com" in full:
                clean_url = "https://www.bbc.com" + full.split("https://www.bbc.com")[-1]
                links.add(normalize_url(clean_url))

    return sorted(links)

def scrape_all_links(category: str, max_days_back: int, max_links: int, wait_s: int):
    all_links = set()

    # Start with today's date
    current_date_obj = datetime.now()

    print(f"Starting link collection for category: {category}")

    # Loop until we have enough links or we've gone back too far
    days_back = 0
    while len(all_links) < max_links and days_back < max_days_back:
        date_str = current_date_obj.strftime("%Y%m%d")
        print(f"Searching archive for: {date_str}...")

        new_links = collect_article_links(date_str, category)
        all_links.update(new_links)

        print(f"Found {len(new_links)} links. Total unique links: {len(all_links)}")

        # Move to the previous day
        current_date_obj -= timedelta(days=1)
        days_back += 1
        time.sleep(wait_s) # Small delay to be polite to the archive

    links_list = sorted(list(all_links))
    
    # Save links to a file
    links_file = f"all_{category}_collected_links.json"
    with open(links_file, "w", encoding="utf-8") as f:
        json.dump(links_list, f, indent=2)
        
    print(f"\nSaved {len(links_list)} unique links to {links_file}")

if __name__ == "__main__":
  scrape_all_links(category="technology", max_days_back=60, max_links=1000, wait_s=15)

Article scraping

In [ ]:
import json
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

BASE = "https://www.bbc.com"
WAYBACK = "https://web.archive.org/web"

def category_from_url(url: str) -> str | None:
    path = urlparse(url).path.strip("/")
    if not path:
        return None
    return path.split("/")[0]

def scrape_article(url: str, category_hint: str | None = None) -> dict:
    # Now fetching DIRECTLY from BBC, not via Wayback
    headers = {"User-Agent": "SchoolAssignmentBot/1.0"}
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "lxml")

    # Title
    h1 = soup.find("h1")
    title = h1.get_text(" ", strip=True) if h1 else None
    if not title and soup.title:
        title = soup.title.get_text(" ", strip=True)

    # Content
    main = soup.find("main")
    scope = main if main else soup
    paragraphs = [p.get_text(" ", strip=True) for p in scope.find_all("p") if len(p.get_text()) > 40]

    cleaned = []
    for t in paragraphs:
        if not cleaned or cleaned[-1] != t:
            cleaned.append(t)

    content = "\n\n".join(cleaned) if cleaned else None

    return {
        "url": url,
        "category": category_hint or category_from_url(url),
        "title": title,
        "content": content,
    }

def build_dataset(input_file: str, category: str, max_articles: int = 30, delay_s: float = 1.0) -> list[dict]:
    # Step 1: Get historical links from Wayback
    with open(input_file, "r", encoding="utf-8") as f:
        urls = json.load(f)
    
    urls = urls[:max_articles]
    print(f"Loaded {len(urls)} links from {input_file}. Starting scrape...")

    dataset = []
    for i, url in enumerate(urls, start=1):
        try:
            # Step 2: Scrape the live version of those links
            item = scrape_article(url, category_hint=category)
            dataset.append(item)
            print(f"[{i}/{len(urls)}] OK - {item['title']}")
        except Exception as e:
            dataset.append({"url": url, "error": str(e)})
            print(f"[{i}/{len(urls)}] Error - {url}")
        time.sleep(delay_s)

    return dataset

if __name__ == "__main__":
    # Historical date to browse, but scraping will be live
    category = "sport"
    filename = "724_all_sport_collected_links.json"
    # Increased max_articles from 5 to 30
    data = build_dataset(filename, category, max_articles=800, delay_s=5)

    out_file = f"bbc_{category}_articles.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"\nSaved {len(data)} items (live content) to {out_file}")

Baseline

In [ ]:
import json
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Download risorse NLTK (solo la prima volta)
nltk.download("punkt")
nltk.download("stopwords")
nltk.download('punkt_tab')

STOP_WORDS = set(stopwords.words("english"))

def load_articles_from_json(path):
    """
    Legge un file JSON e restituisce una lista di articoli
    con category, title e content.
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    articles = []

    for item in data:
        article = {
            "category": item.get("category", ""),
            "title": item.get("title", ""),
            "content": item.get("content", "")
        }
        articles.append(article)

    return articles

def get_common_token(all_tokens_list, max_token=20):
  # Appiattiamo la lista di liste in una singola lista di token
  flattened_tokens = [token for sublist in all_tokens_list for token in sublist]
  
  token_counts = Counter(flattened_tokens)

  # prendi le 10 più frequenti
  top = token_counts.most_common(max_token)

  print("Top 10 parole più frequenti negli articoli:")

  for word, freq in top:
      print(f"{word}: {freq}")
  return top

def count_buzzwords(tokens_article, buzzwords):
    return sum(token in buzzwords for token in tokens_article)

def preprocess_text(text):
    """
    Tokenizza e pulisce il testo:
    - lowercase
    - tokenizzazione
    - rimozione punteggiatura / token non alfabetici
    - rimozione stopwords
    """
    text = text.lower()
    tokens = word_tokenize(text)

    cleaned_tokens = []
    for token in tokens:
        # tiene solo token alfabetici
        if re.fullmatch(r"[a-zA-Z]+", token):
            if token not in STOP_WORDS:
                cleaned_tokens.append(token)

    return cleaned_tokens

def classify_baseline(articles):
  y_true = [article["category"] for article in articles]
  y_pred = [article["prediction"] for article in articles]

  cm = confusion_matrix(y_true, y_pred, labels=["sport", "business"])

  print(cm)
  print(classification_report(y_true, y_pred))
  print("Baseline accuracy:", accuracy_score(y_true, y_pred))
  disp = ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=["sport", "business"],
    display_labels=["sport", "business"],
    cmap="Greens"
  )

  plt.show()

# Caricamento dati (Assicurati che i file esistano nel path)
sport_articles = load_articles_from_json("bbc_sport_articles.json")
business_articles = load_articles_from_json("bbc_business_articles.json")

all_articles = sport_articles + business_articles

baseline_words = ['game', 'league', 'team', 'win', 'season', 'cup', 'players', 'club', 'games', 'time']

# Eseguiamo il preprocessing per ogni articolo
for article in all_articles:
    full_text = article["title"] + " " + article["content"]
    tokens = preprocess_text(full_text)
    article['buzzword_count'] = count_buzzwords(tokens, baseline_words)
    article['prediction'] = 'sport' if article['buzzword_count'] > 10 else 'business'

classify_baseline(all_articles)